# Credit Card Fraud Detection Proof of Concept using Azure ML Studio

## Executive Summary

This document outlines a proof-of-concept (PoC) for a fraud detection workflow, designed for a financial services audience. It demonstrates how Azure Machine Learning (Azure ML) can be used to access a registered transaction dataset, prepare it for analysis, train an anomaly detection model, and review its performance. The PoC also suggests a governance framework for models before they are considered for production use.

The primary business objective is **not** to automate approval or denial decisions with a machine at this stage. Instead, the goal is to showcase how Azure ML can help the company:
*   Identify suspicious transactions more quickly.
*   Prioritize the attention of fraud investigators.
*   Establish a repeatable process for continuous model improvement.

### Key Takeaways for Stakeholders

*   Fraud detection is a "rare event" problem. A model can appear highly accurate overall but still miss a significant number of actual fraudulent transactions.
*   This notebook presents an efficient *screening* workflow, not a complete production-ready fraud control system.
*   The current PoC is most effective as a tool to assist analysts or operate in a "shadow mode" (monitoring without taking action).
*   Decisions about deploying the model should be based on a careful assessment of the business costs associated with false alarms versus the costs of missed fraud, rather than solely on headline accuracy figures.

## End-to-End Pipeline Overview

```text
Source Data (Kaggle Dataset)
        |
        v
Data Stored in Azure ML (Registered Asset)
        |
        v
Analysis Environment (Azure ML Workspace & Notebook)
        |
        v
Prepare Data for Analysis
- Adjust transaction values for fairness
- Remove irrelevant information
- Separate data for model training and testing
        |
        v
Train Anomaly Detection Model (Isolation Forest)
        |
        v
Review Model Performance
- Accuracy of fraud alerts (Precision)
- Ability to catch actual fraud (Recall)
- Unnecessary fraud alerts (False Positives)
- Fraud that was not caught (Missed Fraud)
        |
        v
Save the Trained Model (Model Registry)
        |
        v
Plan for Production Use
- Test in parallel (shadow mode)
- Support human review
- Plan for ongoing checks and updates
```

## Azure ML Components and Their Role

| Azure ML Component            | Role in the Process                                         |
| :---------------------------- | :---------------------------------------------------------- |
| Azure ML Workspace            | Central hub for data, experiments, models, and governance.  |
| Registered Data Asset         | Stores the official, approved transaction data.             |
| Notebook Environment          | Provides a documented space to run and explain the analysis step-by-step. |
| Compute Session or Local Python | Tools to process data and train the model.                  |
| Model Artifact / Registry     | Stores trained model versions for tracking and approval.    |
| Monitoring & Deployment       | Enable future production use, testing, and performance tracking. |

## Workflow

### Step 1: Connect the Notebook to the Required Tools


In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Load the Fraud Dataset from Azure ML

This section imports the approved transaction dataset into our analysis environment from the Azure ML workspace. In business terms, this step signifies the transition from enterprise data storage to the analytics process. Instead of teams sharing local spreadsheet copies, Azure ML maintains a single, registered data source that can be consistently reused across various experiments.

#### Why This Matters

*   **Ensures consistency:** Everyone uses the same, up-to-date data for analysis.
*   **Provides auditability:** Offers a clear audit trail back to the original data source, crucial for compliance.
*   **Improves repeatability:** Allows the analysis to be rerun easily and reliably without rebuilding the data pipeline from scratch.

#### Plain-English Explanation of the Code

*   `Workspace.from_config()`: Connects the notebook to the company's Azure ML workspace.
*   `Dataset.get_by_name(...)`: Retrieves the registered fraud dataset by its name.
*   `.to_pandas_dataframe()`: Converts the dataset into a table format that Python can easily analyze.
*   `df.head()`: Displays the first few rows of the data as a quick check to confirm successful loading.

#### Executive Interpretation

This step is akin to accessing a trusted, official report before starting an analysis. No model is created at this stage. It confirms that the project begins with the correct data in the right setup, which is vital for maintaining compliance and building credibility for the model.


In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Prepare Data for Reliable Analysis

Before training a model, this step standardizes transaction amounts and separates the predictive features from the known fraud labels. This ensures the model compares transactions on a more balanced and relevant scale.

#### Business Reasoning

Models can be misled by uneven data. If one piece of information, like transaction amount, is measured on a much larger scale than others, the model might focus too much on it, potentially missing other important clues. Adjusting transaction amounts helps the model focus on unusual *behavior* patterns rather than just large sums.

#### What the Notebook Is Doing

*   **Adjusts `Amount`:** Transaction amounts are rescaled to prevent extremely large or small purchases from overly influencing the analysis.
*   **Removes `Class` from Inputs:** The `Class` column (which indicates fraud or not) is excluded from the data used to train the model, as it's the answer we are trying to predict.
*   **Excludes `Time`:** The `Time` feature is removed as it's less critical for this analysis compared to the anonymized transaction behavior variables.
*   **Separates Labels (`y`):** The fraud labels are stored separately. This is essential for later evaluating the model's predictions against the actual outcomes.

#### Stakeholder Takeaway

Data preparation is crucial for improving the quality of information *before* the model makes any predictions. Good data preparation is often as important as the choice of model itself, especially in fraud detection where unusual and skewed data are common.


In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Train an Anomaly Detection Model

This proof of concept uses **Isolation Forest**, an anomaly detection technique specifically designed to identify rare and unusual patterns within a large dataset of typical transactions. This is a practical choice for fraud detection because fraud is infrequent, and suspicious activity often stands out from normal behavior.

#### Why This Model Was Chosen

*   **Effective for rare events:** It performs well when fraud cases are uncommon.
*   **Efficient:** It is relatively fast and efficient on large datasets.
*   **Unsupervised:** It does not require the team to manually label every type of suspicious pattern in advance.
*   **Identifies outliers:** It is useful for pinpointing transactions that warrant a closer look by human investigators.

#### What the `contamination` Setting Means

The `contamination` parameter tells the model to expect only a very small percentage of anomalies in the data. This reflects the business reality: most transactions are legitimate, and only a small fraction are fraudulent. This setting helps control how aggressively the model flags potential issues.

#### Stakeholder Takeaway

Isolation Forest is best understood as a *screening tool*. It helps filter a large volume of transactions down to a smaller, more manageable list for review. In a real-world fraud detection system, it would typically be used alongside other methods like specific rules, more advanced models, or human analysis, rather than as a standalone solution.


In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluate the Model in Business Terms

After the model identifies potentially suspicious transactions, we compare its findings against actual fraud records in our data. This comparison helps us determine if this initial proof of concept shows enough promise to warrant further investment.

## Key Metrics

These metrics help us understand not just how often the model is right, but also how effective it is at finding fraud and how much noise it creates.

| Metric | Meaning for the Business |
|---|---|
| Precision | When the model raises a fraud alert, how often is that alert correct? |
| Recall | Of all real fraud cases, how many did the model successfully catch? |
| F1-Score | A combined measure that balances how well the model identifies fraud with how accurate its alerts are. |
| Support | The number of real transactions in each class. |

## Current Proof-of-Concept Result

| Transaction Type | Precision | Recall | F1-Score | Volume |
|---|---:|---:|---:|---:|
| Normal transactions | 1.00 | 1.00 | 1.00 | 284,315 |
| Fraudulent transactions | 0.29 | 0.28 | 0.28 | 492 |

The results show the model is excellent at identifying normal transactions. However, it struggles with detecting fraud.

Specifically, it only catches about **28% of actual fraud cases** (meaning three out of four fraud cases are missed). When it does flag a transaction as fraud, it's correct only about **29% of the time**.

## Why Overall Accuracy Is Misleading

You might see an overall accuracy near 99.9% in the notebook. This high number can be misleading. Because fraud is very rare, a simple model could achieve high accuracy by just labeling almost all transactions as 'normal.' For fraud detection, **focusing on Precision, Recall, and the types of errors (false positives and missed fraud) is far more valuable** than overall accuracy.

## Cost-Benefit View: False Positives vs. Missed Fraud

Understanding the consequences of both types of errors is crucial:

| Outcome | Business Cost |
|---|---|
| False positive | Customer frustration, declined legitimate purchases, contact-center workload, merchant friction, and reputational damage. |
| Missed fraud | Direct financial loss, reimbursement expense, compliance scrutiny, and erosion of trust when fraudulent activity reaches the customer. |

In summary, while **missed fraud typically results in higher direct financial losses per incident**, **too many false positives lead to significant ongoing operational burdens and damage customer relationships.** Therefore, the goal for production isn't simply to generate the most alerts, but to strike a carefully managed balance that aligns with our company's tolerance for risk and our customer service standards.

In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 6: Saving and Preparing the Model for Use

This step involves saving the trained model. This ensures we have a clear record of the model used for testing, allowing us to track its performance and use it for future comparisons. Think of it as creating an official version of our fraud detection tool.

#### Why Saving the Model is Important

*   It establishes a controlled record of the exact model version that was developed.
*   It supports formal review processes and allows easy comparison with any new model versions.
*   It provides a stable, reviewed item for operations and risk teams before it's put into use.

## How to Safely Use the Model

Based on its current performance, this model is **not yet ready** for automatically approving or rejecting transactions. Instead, we recommend using it as a support tool:

1.  **Test in "Shadow Mode":** Run the model alongside your current fraud systems without making automatic decisions.
2.  **Support Analysts:** Use it as a tool to help your fraud review team identify potentially suspicious transactions.
3.  **Review Carefully:** Examine flagged transactions along with your existing fraud checks.
4.  **Track Key Information:** Monitor how many transactions are flagged as suspicious (alerts), how many actual fraud cases are caught, how many legitimate customers are inconvenienced (false positives), and the overall impact on your operations, before considering wider use.

## Recommended Next Steps for Improvement

1.  **Adjust Alert Levels:** Fine-tune the sensitivity of the model's alerts to balance the cost of missing fraud against the inconvenience caused by flagging legitimate transactions.
2.  **Explore Other Models:** Test more advanced models, such as those using gradient boosting techniques, with the existing fraud data.
3.  **Add More Information:** Include more details in the model's analysis, such as the type of business a merchant belongs to, the customer's location, device risk, past transaction history, and how frequently transactions occur.
4.  **Check Performance Across Groups:** Ensure the model performs well for all customer segments and doesn't have weaknesses in specific groups.
5.  **Continuous Improvement:** Set up ongoing checks for changes in fraud patterns, incorporate feedback from human reviewers, and plan for regular model updates.

## Understanding the Risks and How to Manage Them

| Potential Risk | Why it Matters | How to Manage It |
|---|---|---|
| **Too many false positives** (flagging good customers) | Can annoy or block legitimate customers, leading to a bad experience. | Start with the model in testing mode, carefully set alert thresholds, and monitor how often flagged transactions are confirmed as suspicious. |
| **Too many missed fraud cases** (not catching actual fraud) | Fraud losses continue, making the model seem effective when it's not catching real threats. | Closely track the model's success in identifying actual fraud and compare its performance against current detection methods. |
| **Data drift over time** (fraud patterns change) | Fraudsters adapt, and an older model can become less effective quickly. | Regularly check the model's performance, update it with recent data, and monitor changes in transaction patterns. |
| **Limited explainability** (hard to understand *why* a transaction is flagged) | May lead to a lack of trust from stakeholders or difficulty in approving its use. | Combine the model's alerts with explanations of *why* it flagged a transaction (using tools like SHAP) and add notes from human reviewers. |
| **Compliance and fairness concerns** (inconsistent or biased outcomes) | Can lead to legal issues and damage the company's reputation. | Document all decision-making processes, test performance across different customer groups, and involve legal and compliance teams before deployment. |

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Understanding the Volume of Alerts

This chart helps answer a key operational question: **Approximately how many suspicious transactions will our review team need to examine?**

#### How to Interpret This Chart

*   The bar labeled `0` shows transactions the model identifies as normal.
*   The bar labeled `1` shows transactions the model flags as suspicious or potentially fraudulent.

#### What This Means for the Business

*   It's normal for the number of actual fraud cases (bar `1`) to be much smaller than normal transactions (bar `0`), as fraud is typically rare.
*   If the number of flagged transactions (bar `1`) is too high, our review team could become overwhelmed.
*   If the number of flagged transactions is too low, the model might be too cautious and miss actual fraud.

This visualization provides an early indication of staffing needs and workload for the fraud operations team, helping to estimate if the alert volume is manageable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Step 7 (Continued): Examining Transaction Values for Flagged Items

This chart compares the transaction amounts for normal versus suspicious transactions. This is important because a fraud detection system should ideally identify suspicious activity based on a combination of factors, not just solely on large transaction values.

#### What This Chart Shows

*   The central line within each box represents the typical transaction amount for that group.
*   Wider boxes or longer "tails" indicate a greater range or spread in transaction amounts.
*   Individual points outside the main boxes represent transactions with unusually high or low amounts compared to the typical range.

#### Business Interpretation

If the transactions flagged as suspicious are mostly concentrated around very high dollar amounts, the model might be too narrowly focused. Effective fraud detection often involves recognizing patterns beyond just high values, such as:

*   Multiple smaller transactions occurring in quick succession.
*   A customer suddenly using different payment methods or channels.
*   Unusual sequences of customer activity.

This view helps us confirm if the model is considering broader behavioral patterns, not just transaction value.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Step 7 (Continued): Understanding *Why* the Model Flags Transactions

This chart (a SHAP beeswarm plot) helps us understand the reasons behind the model's decisions. It shows which factors (or "features") were most important in determining whether a transaction was flagged as suspicious.

#### Why This is Important for Decision-Makers

*   It enables our analysts to explain to customers or stakeholders *why* a specific transaction was flagged.
*   It builds confidence and trust in the initial results of this fraud detection test.
*   It guides us on which factors to focus on when improving the model in the future.

#### How to Read This Chart (Simple Guide)

*   Each row on the chart represents a specific piece of information the model uses (e.g., transaction amount, customer location).
*   Factors listed higher up on the chart had a greater impact on the model's decision.
*   Dots to the right of the center line indicate that this factor pushed the transaction towards being flagged as suspicious.
*   Red dots generally represent higher values for that factor, while blue dots represent lower values.

## Communicating About the Model's Current Capabilities and Limitations

When discussing this initial fraud detection test with executives and teams, it's important to convey a consistent message:

1.  **This is a Support Tool:** The model is a promising *early warning system*, but it's not yet reliable enough for fully automated decision-making.
2.  **Report Key Metrics:** Always include both the rate at which actual fraud is caught and the rate at which legitimate transactions are incorrectly flagged.
3.  **Accuracy vs. Quality:** A high overall accuracy score doesn't automatically mean the model is good at catching fraud.
4.  **Show Examples:** Use specific examples from this SHAP chart and feedback from investigators to illustrate why transactions are flagged.
5.  **Phased Rollout:** Only consider expanding the model's use after demonstrating its effectiveness in a testing phase, showing it catches more fraud with acceptable customer impact.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

## Recommendation for Leadership

This notebook outlines a reliable process for setting up a fraud detection test using Azure ML. However, the current model should be viewed as a **tool to assist decision-making**, rather than an automated system for final control.

The recommended next step for the business is to conduct a controlled pilot program. This pilot should involve:

*   **Human Oversight:** Fraud review specialists should examine the transactions flagged by the model.
*   **Enhanced Data:** Use more detailed and relevant data points to improve the model's accuracy.
*   **Defined Rules:** Establish clear criteria for what constitutes a suspicious transaction that warrants review.

This approach will help protect our customers, set realistic expectations about the current model's capabilities, and provide a clear, practical path for moving from this initial test to a fully functional production system.